# NYC 311 AutoDDG: milestone notebook

This notebook is the presentation of the project. Run it from top to bottom. Every milestone ends with an **Outcome** section: the table or file that goes into the report.

**Question.** How well can an AutoDDG-style description, grounded in an Apache Spark profile, describe NYC 311 service requests and make that dataset easier to find than the short portal text?

**Data.**
- Fact table: [311 Service Requests from 2020 to Present](https://data.cityofnewyork.us/d/erm2-nwe9), id `erm2-nwe9`.
- Dimension table: [ZIP Code Tabulation Areas](https://data.cityofnewyork.us/d/35j5-n34v), id `35j5-n34v` (221 rows).

The default run is a **dated sample**, not the full archive. The profile records the window, and the generated text is required to say so.


## Before you run anything

1. Use **Python 3.11** and **JDK 17**. PySpark will not start on a machine with no `java` on `PATH`.
2. From the repo root: `python3.11 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt`.
3. `cp .env.example .env` and set `OPENAI_API_KEY`. Milestones 1, 2, and the human BM25 index run without it. Description generation and the judge do not.
4. Leave the default window (`2025-06-01` inclusive to `2025-06-03` exclusive, at most 20,000 rows) until this notebook finishes once.
5. Start Jupyter with the repo root as the working directory, or start it from `notebooks/`. The setup cell handles both.

Do not commit `.env`, the CSV pages, or the Parquet output.


In [1]:
!pip install -r ../requirements.txt


[notice] A new release of pip is available: 26.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import dotenv
import os

from autoddg import AutoDDG

dotenv.load_dotenv()

print(os.getenv("OPENAI_API_KEY"))

/Users/lvcaitong/Desktop/NYU/Big data/Big Data final/.venv/lib/python3.11/site-packages/autoddg/llm/__init__.py:4: UserWarning: transformers and torch not available. Install with: pip install transformers torch
  from .local import LocalLLMClient


## How these steps map onto the proposal

The proposal's four milestones are the grading structure. This notebook uses the same four numbers.

| Proposal milestone | What this notebook finishes |
| --- | --- |
| 1. Setup, baselines, Spark | NYC Open Data lake, raw audit, and one Auctus table with a human description |
| 2. Generation and Spark patterns | ZIP/ZCTA join, profiles, AutoDDG SFD/UFD for 311 and Auctus, and OpenAI scores against the human text |
| 3. Evaluation | BM25 MRR and Recall@1 for portal text versus the 311 SFD |
| 4. Report, due 9 December | ACM-style paper from the outcome files, plus the GitHub link |

Stretch work from the proposal (a dataset with no description, a second judge, multimodal data) stays after this notebook.


## Setup check


In [3]:
import shutil
import sys
from pathlib import Path

#hello this has been updated

import os

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display

from src import config


def show_table(title, rows, columns=None, limit=20):
    display(Markdown(f"### {title}"))
    rows = list(rows)[:limit]
    if not rows:
        display(Markdown("_No rows._"))
        return
    columns = columns or list(rows[0].keys())
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for row in rows:
        cells = [str(row.get(column, "")).replace("|", "/") for column in columns]
        lines.append("| " + " | ".join(cells) + " |")
    display(Markdown("\n".join(lines)))


java = shutil.which("java")
display(Markdown(
    f"""### Environment
| Item | Value |
| --- | --- |
| Repo | `{config.ROOT}` |
| Sample window | `{config.START_DATE}` inclusive, `{config.END_DATE}` exclusive |
| Row cap | `{config.SAMPLE_ROWS}` (`0` means the whole window) |
| Java on PATH | `{java or "MISSING — install Temurin 17 before Milestone 1 Spark"}` |
| OpenAI key loaded | `{bool(os.getenv("OPENAI_API_KEY"))}` |
| Model | `{config.OPENAI_MODEL}` |
"""
))


### Environment
| Item | Value |
| --- | --- |
| Repo | `/Users/lvcaitong/Desktop/NYU/Big data/Big Data final` |
| Sample window | `2025-06-01` inclusive, `2025-06-03` exclusive |
| Row cap | `20000` (`0` means the whole window) |
| Java on PATH | `/usr/bin/java` |
| OpenAI key loaded | `False` |
| Model | `gpt-4o` |


## Milestone 1 — Ingestion, storage, and raw profiling

**Goal.** Land a reproducible slice of both tables and record the schema, key uniqueness, and missingness that later stages are allowed to talk about.

**Do these steps**

1. Create the lake directories (the download functions do this).
2. Download 311 from the SODA CSV endpoint in pages of at most 50,000 rows. A single `$limit=500000` request is not a reliable way to get half a million rows. The code orders by `unique_key` inside the date window so the same `.env` reproduces the same pages.
3. Download ZCTA centroids only. Skip `the_geom`. The dimension table is the broadcast side, and the polygon column is not used by the ZIP equality join.
4. Save the official portal name and description for 311 and four other datasets. Those descriptions are the human baseline. Fetch them here, before any model text exists.
5. Start Spark locally. Read every column as a string. Inferring types would turn `incident_zip` into an integer and drop leading zeros.
6. Count rows, distinct `unique_key`, and null or blank values on the join and category fields. Classify columns as identifier, temporal, spatial, text, or categorical.

**Outcome files:** `data/raw/fact_311/`, `data/raw/dim_zcta/`, `data/processed/catalog.json`, `data/processed/m1_raw_audit.json`.


In [4]:
from src.ingest import download_fact_311, download_zcta, fetch_catalog

fact_manifest = download_fact_311()
zcta_manifest = download_zcta()
catalog = fetch_catalog()

show_table(
    "Download manifest",
    [
        {"table": "311 fact", **{k: fact_manifest[k] for k in ("dataset_id", "rows", "pages", "where")}},
        {"table": "ZCTA dimension", **{k: zcta_manifest[k] for k in ("dataset_id", "rows", "pages", "select")}},
    ],
)
show_table(
    "Portal catalog (human baselines)",
    [
        {
            "id": row["id"],
            "name": row["name"],
            "columns": row["n_columns"],
            "description": (row["human_description"][:180] + "…") if len(row["human_description"]) > 180 else row["human_description"],
        }
        for row in catalog
    ],
)


### Download manifest

| table | dataset_id | rows | pages | where |
| --- | --- | --- | --- | --- |
| 311 fact | erm2-nwe9 | 19522 | 1 | created_date >= '2025-06-01T00:00:00.000' AND created_date < '2025-06-03T00:00:00.000' |
| ZCTA dimension | 35j5-n34v | 221 | 1 |  |

### Portal catalog (human baselines)

| id | name | columns | description |
| --- | --- | --- | --- |
| erm2-nwe9 | 311 Service Requests from 2020 to Present | 48 | NOTE: Learn more about the latest changes to this dataset: https://opendata.cityofnewyork.us/311-service-requests-from-2010-to-present-updates/ 311 responds to thousands of inquiri… |
| h9gi-nx95 | Motor Vehicle Collisions - Crashes | 29 | The Motor Vehicle Collisions crash table contains details on the crash event. Each row represents a crash event. The Motor Vehicle Collisions data tables contain information from a… |
| 43nn-pn8j | DOHMH New York City Restaurant Inspection Results | 31 | The dataset contains every sustained or not yet adjudicated violation citation from every full or special program inspection conducted up to three years prior to the most recent in… |
| 5uac-w243 | NYPD Complaint Data Current (Year To Date) | 41 | This dataset includes all valid felony, misdemeanor, and violation crimes reported to the New York City Police Department (NYPD) for all complete quarters so far this year (2019). … |
| 35j5-n34v | ZIP Code Tabulation Areas | 8 | This is a subset of ZIP Code Tabulation Areas or ZCTAs (pronounced zik-tahs) that fall within New York City. ZCTAs are a geographic product of the U.S. Census Bureau created to all… |
| datamart.socrata.data-cdc-gov.kee5-23sr | PLACES: ZCTA Data (GIS Friendly Format), 2024 release | 84 | This dataset contains model-based ZIP Code Tabulation Area (ZCTA) level estimates in GIS-friendly format. PLACES covers the entire United States—50 states and the District of Colum… |

### Auctus table that joins on ZIP

CDC PLACES 2024 ZCTA estimates are indexed by Auctus. `ZCTA5` matches `clean_zip` on the 311 fact table. The download writes `data/raw/auctus_places_zcta/part-00001.csv` and appends the dataset to `data/processed/catalog.json`.


In [5]:
from src.ingest import download_auctus_places

auctus_record = download_auctus_places()
show_table(
    "Outcome — Auctus catalog entry",
    [{
        "id": auctus_record["id"],
        "name": auctus_record["name"],
        "n_columns": auctus_record["n_columns"],
        "join": "clean_zip = ZCTA5",
        "description": auctus_record["human_description"][:180] + "…",
    }],
)


### Outcome — Auctus catalog entry

| id | name | n_columns | join | description |
| --- | --- | --- | --- | --- |
| datamart.socrata.data-cdc-gov.kee5-23sr | PLACES: ZCTA Data (GIS Friendly Format), 2024 release | 84 | clean_zip = ZCTA5 | This dataset contains model-based ZIP Code Tabulation Area (ZCTA) level estimates in GIS-friendly format. PLACES covers the entire United States—50 states and the District of Colum… |

### Spark load and structural audit

If the next cell says Java is missing, install it, export `JAVA_HOME`, restart the kernel, and run from the setup cell again. A Spark session that was created earlier in the same kernel keeps its old configuration, so restart rather than calling `getOrCreate` a second time after changing memory settings.


In [6]:
from src.config import RAW_311

print(RAW_311)

parts = sorted(str(path.resolve().as_uri()) for path in RAW_311.glob("part-*.csv"))
print(parts)

/Users/lvcaitong/Desktop/NYU/Big data/Big Data final/data/raw/fact_311
['file:///Users/lvcaitong/Desktop/NYU/Big%20data/Big%20Data%20final/data/raw/fact_311/part-00001.csv']


In [ ]:
if shutil.which("java") is None:
    raise RuntimeError(
        "Java is not on PATH. Install Temurin 17, export JAVA_HOME, restart the kernel, and rerun from the setup cell."
    )

from src.spark_etl import audit_raw, load_fact, load_zcta, start_spark

spark = start_spark()
fact_raw = load_fact(spark)
zcta_raw = load_zcta(spark)
audit = audit_raw(fact_raw)

display(Markdown(
    f"""### Outcome — raw 311 audit
| Check | Value |
| --- | --- |
| Rows in this sample | {audit['total_rows']:,} |
| Distinct `unique_key` | {audit['unique_keys']:,} |
| Duplicate keys | {audit['duplicate_keys']:,} |
| Columns | {audit['n_columns']} |
| ZCTA rows loaded | {zcta_raw.count():,} |
| Window | {audit['sample_window']['start']} to {audit['sample_window']['end']} (end exclusive) |
"""
))
show_table(
    "Null or blank counts on fields the join and the description depend on",
    [{"column": name, "null_or_blank": n} for name, n in audit["null_or_blank_counts"].items()],
)
roles = {}
for column in audit["columns"]:
    roles.setdefault(column["role"], []).append(column["name"])
show_table(
    "Column roles",
    [{"role": role, "columns": ", ".join(names)} for role, names in sorted(roles.items())],
)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/17 14:37:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Milestone 1 is done when

- The fact manifest row count is greater than zero and not larger than `SAMPLE_ROWS`.
- `duplicate_keys` is zero. Investigate before joining if it is not.
- `incident_zip`, `created_date`, and `complaint_type` show their real missingness. Those counts are ground truth for the judge later.
- The catalog has a non-empty human description for every id that the search queries name. If a portal id was retired, remove that id from `CATALOG_IDS` and `SEARCH_QUERIES` in `src/config.py` together, and write down the change in the report.


## Milestone 2 — Cleaning, generation, and baseline scoring

**Goal.** Produce the enriched 311 table, generate AutoDDG descriptions for that table and for the Auctus PLACES table, and score both generated texts and the human-authored descriptions on faithfulness, completeness, and searchability.

**Do these steps**

1. Build `clean_zip` from the digits in `incident_zip`. Keep the first five. ZIP+4 values such as `10001-1234` become `10001`.
2. Parse `created_date` and `closed_date` with both timestamp styles. The SODA file is `yyyy-MM-dd'T'HH:mm:ss.SSS`. A bulk portal export is `MM/dd/yyyy hh:mm:ss a`.
3. Drop rows whose ZIP is not five digits or whose created timestamp did not parse. Record how many rows each rule removes.
4. Broadcast the ZCTA table and left-join on `clean_zip = dim_zip`. Report the match rate. A USPS ZIP and a Census ZCTA are not the same geography.
5. Compute median hours to close for the top complaint types, then write Parquet partitioned by `year` and `month`.
6. Build a Spark profile for the enriched 311 sample and for the Auctus PLACES ZCTA table. The profile is the only evidence the model may cite.
7. Generate an SFD and a UFD for each profile.
8. Run the same judge on the generated text and on the human portal description. The profile is the ground truth for both scores.

The judge compares each text with the profile of the table you actually analyzed. The 311 portal description covers the full archive, so it can score lower on faithfulness for this two-day sample.


In [ ]:
from src.spark_etl import (
    apply_fact_filters,
    broadcast_join,
    clean_fact,
    clean_zcta,
    join_report,
    partition_counts,
    resolution_pattern,
    write_curated,
)

fact_clean = clean_fact(fact_raw)
fact_kept, cleaning = apply_fact_filters(fact_clean)
zcta_clean = clean_zcta(zcta_raw)
joined = broadcast_join(fact_kept, zcta_clean)
joined.cache()
match = join_report(joined)
resolution = resolution_pattern(joined)
curated_path = write_curated(joined)
partitions = partition_counts(spark)

show_table("Outcome — rows removed by cleaning", [cleaning])
display(Markdown(
    f"""### Outcome — broadcast join
| Check | Value |
| --- | --- |
| Rows after cleaning | {match['rows']:,} |
| Matched to a ZCTA | {match['matched_rows']:,} |
| Unmatched | {match['unmatched_rows']:,} |
| Match rate | {match['match_rate']:.1%} |
| Curated Parquet | `{curated_path}` |
"""
))
display(Markdown(match["note"]))
show_table("Most common ZIPs with no ZCTA match", match["top_unmatched_zips"])
show_table("Outcome — median hours to close, top complaint types", resolution)
show_table("Outcome — Parquet partitions", partitions)


### AutoDDG descriptions

`build_profile` summarizes the enriched 311 sample. `build_auctus_profile` summarizes the full PLACES ZCTA table. `generate_descriptions` writes an SFD and a UFD from that profile only.


In [ ]:
from src.autoddg_engine import generate_descriptions
from src.config import PROCESSED
from src.profile import build_profile

profile = build_profile(
    joined,
    match,
    resolution,
    is_sample=config.SAMPLE_ROWS != 0,
)
descriptions = generate_descriptions(profile, output_path=PROCESSED / "autoddg_descriptions.json")
display(Markdown("### 311 search-focused description\n\n" + descriptions["sfd"]))
display(Markdown("### 311 user-focused description\n\n" + descriptions["ufd"]))


In [ ]:
from src.profile import build_auctus_profile, load_auctus_frame

auctus_frame = load_auctus_frame(spark)
auctus_profile = build_auctus_profile(auctus_frame)
auctus_descriptions = generate_descriptions(
    auctus_profile,
    output_path=PROCESSED / "autoddg_descriptions_auctus.json",
)
display(Markdown("### Auctus search-focused description\n\n" + auctus_descriptions["sfd"]))
display(Markdown("### Auctus user-focused description\n\n" + auctus_descriptions["ufd"]))


In [ ]:
from src.autoddg_engine import human_description, score_against_baseline, write_scorecard
from src.config import AUCTUS_ID, FACT_ID

scorecard = write_scorecard([
    score_against_baseline(profile, descriptions, human_description(FACT_ID), "311"),
    score_against_baseline(
        auctus_profile,
        auctus_descriptions,
        human_description(AUCTUS_ID),
        "auctus",
    ),
])
show_table(
    "Outcome — baseline scores against the Spark profile",
    scorecard["rows"],
    ["dataset", "text", "faithfulness", "completeness", "searchability"],
)
display(Markdown("Saved to `data/processed/eval_judge_comparison.json`."))


### Milestone 2 is done when

- The cleaning report, join match rate, and Parquet partition count are saved.
- `content_profile.json` and `content_profile_auctus.json` exist.
- Both description files exist, and every number in them appears in the matching profile.
- `eval_judge_comparison.json` has four rows: generated and human for 311, and generated and human for the Auctus table.


## Milestone 3 — Retrieval benchmark

Description generation and the OpenAI baseline scores are in Milestone 2. This milestone asks whether the 311 SFD is easier to retrieve than the portal description.

One document per catalog dataset. Queries stay frozen in `src/config.py`. Report MRR and Recall@1. The second index replaces only the 311 document with its SFD.


### BM25

Do not compare one raw BM25 score on a short portal sentence with one score on a long SFD. Index one document per catalog dataset, keep the queries in `src/config.py` frozen, and report MRR and Recall@1. The second index replaces only the 311 document with its SFD.


In [ ]:
from src.search_eval import evaluate_retrieval

retrieval = evaluate_retrieval()
human = retrieval["human_portal_index"]
sfd_index = retrieval["autoddg_sfd_index"]
display(Markdown(
    f"""### Outcome — retrieval
| Index | Documents | Queries | MRR | Recall@1 |
| --- | --- | --- | --- | --- |
| Portal descriptions | {human['n_documents']} | {human['n_queries']} | {human['mrr']} | {human['recall_at_1']} |
| 311 document replaced by SFD | {sfd_index['n_documents'] if sfd_index else "—"} | {sfd_index['n_queries'] if sfd_index else "—"} | {sfd_index['mrr'] if sfd_index else "run Milestone 3"} | {sfd_index['recall_at_1'] if sfd_index else "run Milestone 3"} |
"""
))
show_table(
    "Rank of the correct dataset under portal text",
    human["queries"],
    ["query", "relevant_id", "rank", "top_id"],
)
if sfd_index:
    show_table(
        "Rank of the correct dataset after the 311 SFD swap",
        sfd_index["queries"],
        ["query", "relevant_id", "rank", "top_id"],
    )
display(Markdown("Saved to `data/processed/eval_bm25.json`."))


## Milestone 4 — Report

The faithfulness, completeness, and searchability comparison is already in Milestone 2 (`data/processed/eval_judge_comparison.json`). Use that table, the two description files, the join match rate, and the BM25 table in the paper.

### What the proposal still requires

1. **Missing dictionary.** Profile one NYC Open Data or Auctus dataset whose description is empty, and generate an SFD/UFD for it. Keep those numbers separate from the 311 and PLACES scores.
2. **Scale.** Widen `START_DATE` / `END_DATE` and set `SAMPLE_ROWS=0` only after this sample run is the one you intend to cite.
3. **Second judge and multimodal work.** Only after the items above.
4. Publish the repo and put its URL in the ACM-style report. Map the outcome files into Introduction, Problem formulation, Related work, Methods, Results, and References.
